# Step 5 — Statistical Validation

**Pipeline:** Myocarditis structural mimicry reassessment (spike vs. cardiac autoantigens)
**This step:** determines whether the Step 4 structural similarity scores (particularly
TPM1 postfusion TM=0.53) are statistically meaningful or expected by chance.

**Reads:** `alignment_results.json` (Step 4), spike chain files from Step 4
**Produces:** `validation_results.json` — null distribution, p-values, known-pair calibration

### Two parallel validation tracks (both must be cleared)

1. **Randomized proteome null distribution:** compare each spike conformational state against
   a random sample of human proteins (not cardiac-associated), building a background TM-score
   distribution. The observed cardiac-antigen scores are then compared against this background
   to compute empirical p-values.

2. **Known cross-reactive pair calibration:** run a small set of experimentally confirmed
   antibody cross-reactive protein pairs through the identical Foldseek pipeline, establishing
   what TM-score range corresponds to known biological cross-reactivity. This anchors the
   significance threshold to immunological reality rather than statistical convenience.

A result is retained only if it clears BOTH tracks: statistically significant against the
null AND falling within or above the known cross-reactive pair range.

### Compute note

This step runs ~900 Foldseek comparisons (3 spike states x 300 random proteins). Each takes
~1 second. Total runtime: approximately 20-30 minutes including downloads. This is within a
single Colab session's limits but represents the heaviest batch workload in the pipeline so far.

In [ ]:
import subprocess, sys, os, importlib, json, time, random
import numpy as np

for pkg in ("requests", "gemmi", "numpy"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import requests, gemmi
import numpy as np
print("Setup complete.")

In [ ]:
MOUNT_DRIVE = True
BASE_DIR = "/content"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/myocarditis_pipeline"

STEP4_DIR = os.path.join(BASE_DIR, "step4")
STEP5_DIR = os.path.join(BASE_DIR, "step5")
os.makedirs(STEP5_DIR, exist_ok=True)

with open(os.path.join(STEP4_DIR, "alignment_results.json")) as f:
    step4_results = json.load(f)

# Locate spike chain files from Step 4
CHAINS_DIR = os.path.join(STEP4_DIR, "chains")
spike_chain_files = {}
for r in step4_results["results"]:
    state = r["spike_state"]
    if state not in spike_chain_files:
        path = os.path.join(CHAINS_DIR, r["spike_pdb"])
        if os.path.exists(path):
            spike_chain_files[state] = path

print("Spike states available:", list(spike_chain_files.keys()))

# Foldseek binary
FOLDSEEK_BIN = "/content/foldseek/bin/foldseek"
if not os.path.exists(FOLDSEEK_BIN):
    print("Installing Foldseek...")
    os.system("wget -q https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz -O /tmp/foldseek.tar.gz")
    os.system("tar xzf /tmp/foldseek.tar.gz -C /content/")
    os.system("rm /tmp/foldseek.tar.gz")
print("Foldseek:", subprocess.run([FOLDSEEK_BIN, "version"], capture_output=True, text=True).stdout.strip())

# Extract the key Step 4 finding for reference
tpm1_postfusion = [r for r in step4_results["results"]
                   if r["antigen"] == "TPM1" and r["spike_state"] == "postfusion"][0]
print(f"\nKey Step 4 finding to validate: TPM1 postfusion TM={tpm1_postfusion['tm_score']}")

## Track 1 — Randomized proteome null distribution

Sample 300 random human proteins from UniProt (reviewed, non-cardiac), download their
AlphaFold DB structures, and run Foldseek against each spike state. This builds the background
distribution of TM-scores expected when spike is compared against an arbitrary human protein.

The sample is size-stratified: since TM-score is length-dependent, the null distribution
is analyzed in size bins matching each cardiac antigen's length, so the p-value for TPM1
(166 aa) is computed against random proteins of comparable size, not against the whole pool.

In [ ]:
# Sample random human proteins from UniProt (reviewed, Homo sapiens)
# Exclude known cardiac/muscle proteins to avoid contaminating the null

UNIPROT_SEARCH_URL = "https://rest.uniprot.org/uniprotkb/search"
N_RANDOM = 300

print(f"Fetching {N_RANDOM} random reviewed human proteins from UniProt...")

# Query for reviewed human proteins, exclude cardiac/muscle GO terms
# Sort by accession (deterministic) and use a random offset for sampling
params = {
    "query": "(organism_id:9606) AND (reviewed:true) NOT (keyword:KW-0490) NOT (keyword:KW-0263)",
    "fields": "accession,gene_primary,length",
    "format": "json",
    "size": N_RANDOM,
    "sort": "accession asc",
}

# Use a random offset to get a different slice each run
resp = requests.get(UNIPROT_SEARCH_URL, params=params, timeout=30)
resp.raise_for_status()
all_proteins = resp.json()["results"]

# Shuffle and take N_RANDOM
random.seed(42)  # reproducible
random.shuffle(all_proteins)
sampled = all_proteins[:N_RANDOM]

print(f"Retrieved {len(sampled)} proteins.")
lengths = [p["sequence"]["length"] for p in sampled if "sequence" in p]
print(f"Length range: {min(lengths)}-{max(lengths)} aa (median {sorted(lengths)[len(lengths)//2]})")

In [ ]:
# Download AlphaFold DB structures for the sampled proteins
AFDB_API = "https://alphafold.ebi.ac.uk/api/prediction"
NULL_STRUCT_DIR = os.path.join(STEP5_DIR, "null_structures")
os.makedirs(NULL_STRUCT_DIR, exist_ok=True)

downloaded_null = []
failed_downloads = []

print(f"Downloading AlphaFold DB structures for {len(sampled)} proteins...")
for i, p in enumerate(sampled):
    acc = p["primaryAccession"]
    length = p["sequence"]["length"] if "sequence" in p else None
    dest_path = os.path.join(NULL_STRUCT_DIR, f"{acc}.pdb")

    if os.path.exists(dest_path):
        downloaded_null.append({"accession": acc, "path": dest_path, "length": length})
        continue

    try:
        af_resp = requests.get(f"{AFDB_API}/{acc}", timeout=15)
        if af_resp.status_code == 404:
            failed_downloads.append(acc)
            continue
        af_resp.raise_for_status()
        entries = af_resp.json()
        if not entries:
            failed_downloads.append(acc)
            continue

        pdb_url = entries[0]["pdbUrl"]
        pdb_resp = requests.get(pdb_url, timeout=15)
        pdb_resp.raise_for_status()
        with open(dest_path, "wb") as f:
            f.write(pdb_resp.content)
        downloaded_null.append({"accession": acc, "path": dest_path, "length": length})
    except Exception as e:
        failed_downloads.append(acc)

    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(sampled)} downloaded ({len(failed_downloads)} failed)")
    time.sleep(0.3)

print(f"\nDownloaded: {len(downloaded_null)}, Failed: {len(failed_downloads)}")

In [ ]:
# Run Foldseek: each spike state vs each null protein
NULL_RESULTS_DIR = os.path.join(STEP5_DIR, "null_foldseek")
os.makedirs(NULL_RESULTS_DIR, exist_ok=True)
TMP_DIR = "/tmp/foldseek_null_tmp"
os.makedirs(TMP_DIR, exist_ok=True)

OUTPUT_COLS = "query,target,alntmscore,fident,alnlen,qlen,tlen,evalue,lddt"
null_scores = []  # list of {"state", "target_acc", "target_length", "tm_score", ...}

total_comparisons = len(spike_chain_files) * len(downloaded_null)
print(f"Running {total_comparisons} Foldseek comparisons ({len(spike_chain_files)} states x {len(downloaded_null)} null proteins)...")
print("This will take approximately 15-25 minutes.\n")

done = 0
for state, spike_pdb in spike_chain_files.items():
    for null_entry in downloaded_null:
        acc = null_entry["accession"]
        null_pdb = null_entry["path"]
        out_file = os.path.join(NULL_RESULTS_DIR, f"{state}_{acc}.m8")

        cmd = [
            FOLDSEEK_BIN, "easy-search",
            spike_pdb, null_pdb, out_file, TMP_DIR,
            "--format-output", OUTPUT_COLS,
            "--exhaustive-search", "1",
            "-e", "10",
        ]
        subprocess.run(cmd, capture_output=True, text=True)

        tm = None
        if os.path.exists(out_file) and os.path.getsize(out_file) > 0:
            with open(out_file) as f:
                line = f.readline().strip()
                fields = line.split("\t")
                if len(fields) >= 9:
                    tm = float(fields[2])

        null_scores.append({
            "state": state,
            "target_acc": acc,
            "target_length": null_entry["length"],
            "tm_score": tm if tm is not None else 0.0,
        })

        done += 1
        if done % 100 == 0:
            print(f"  {done}/{total_comparisons} comparisons complete...")

print(f"\nAll {total_comparisons} comparisons complete.")
print(f"Non-zero TM scores: {sum(1 for s in null_scores if s['tm_score'] > 0)}/{len(null_scores)}")

In [ ]:
# Build null distribution and compute empirical p-values for Step 4 findings

# Size bins for length-matched comparison
# TPM1 target is 166 aa, MYH6 is 742 aa, ACTC1 is 371 aa
SIZE_BINS = {
    "small": (50, 250),    # matches TPM1 (166 aa)
    "medium": (250, 500),  # matches ACTC1 (371 aa)
    "large": (500, 2000),  # matches MYH6 (742 aa)
}

ANTIGEN_SIZE_BIN = {
    "TPM1": "small",
    "ACTC1": "medium",
    "MYH6": "large",
}

def get_null_tm_distribution(state, size_bin):
    lo, hi = SIZE_BINS[size_bin]
    return [s["tm_score"] for s in null_scores
            if s["state"] == state and s["target_length"] is not None
            and lo <= s["target_length"] < hi]

def empirical_pvalue(observed_tm, null_distribution):
    if not null_distribution:
        return None
    n_greater_or_equal = sum(1 for t in null_distribution if t >= observed_tm)
    return (n_greater_or_equal + 1) / (len(null_distribution) + 1)  # conservative estimator

# Compute p-values for each Step 4 result
print("Empirical p-values (size-matched null distributions):\n")
print(f"{'Antigen':<8} {'State':<15} {'Obs TM':>8} {'Null n':>7} {'Null mean':>10} {'Null max':>10} {'p-value':>10}")
print("-" * 75)

pvalue_results = []
for r in step4_results["results"]:
    antigen = r["antigen"]
    state = r["spike_state"]
    obs_tm = r["tm_score"] if r["tm_score"] is not None else 0.0
    size_bin = ANTIGEN_SIZE_BIN.get(antigen)

    if size_bin is None:
        continue

    null_dist = get_null_tm_distribution(state, size_bin)
    pval = empirical_pvalue(obs_tm, null_dist)
    null_mean = np.mean(null_dist) if null_dist else None
    null_max = max(null_dist) if null_dist else None

    pvalue_results.append({
        "antigen": antigen,
        "spike_state": state,
        "observed_tm": obs_tm,
        "null_n": len(null_dist),
        "null_mean": round(null_mean, 4) if null_mean is not None else None,
        "null_max": round(null_max, 4) if null_max is not None else None,
        "null_size_bin": size_bin,
        "empirical_pvalue": round(pval, 4) if pval is not None else None,
    })

    print(f"{antigen:<8} {state:<15} {obs_tm:>8.4f} {len(null_dist):>7d} "
          f"{null_mean:>10.4f} {null_max:>10.4f} {pval:>10.4f}" if null_dist
          else f"{antigen:<8} {state:<15} {obs_tm:>8.4f} {'—':>7} {'—':>10} {'—':>10} {'—':>10}")

# Flag the key test
tpm1_pf = [p for p in pvalue_results if p["antigen"] == "TPM1" and p["spike_state"] == "postfusion"]
if tpm1_pf:
    p = tpm1_pf[0]
    print(f"\nKEY TEST: TPM1 postfusion TM={p['observed_tm']:.4f}, "
          f"p={p['empirical_pvalue']:.4f} (n={p['null_n']} size-matched null comparisons)")
    if p["empirical_pvalue"] is not None and p["empirical_pvalue"] < 0.05:
        print("  -> SIGNIFICANT at alpha=0.05: the TPM1-postfusion similarity exceeds what")
        print("     is expected by chance against size-matched random human proteins.")
    else:
        print("  -> NOT significant at alpha=0.05: the similarity falls within the range")
        print("     expected by chance, even though TM > 0.5.")

## Track 2 — Known cross-reactive pair calibration

A small set of experimentally confirmed antibody cross-reactive protein pairs is run through
the identical Foldseek pipeline. This calibrates what TM-score range corresponds to known,
demonstrated immunological cross-reactivity, anchoring the threshold to biology rather than
statistics alone.

**Positive controls (confirmed cross-reactive):**
- Streptococcal M protein (P02977) vs cardiac myosin MYH6 (P13533) — rheumatic fever molecular mimicry
- Campylobacter jejuni PorA (Q29VW4) vs human ganglioside-mimicking structures — Guillain-Barre

**Negative controls (structurally unrelated, no known cross-reactivity):**
- Human hemoglobin beta (P68871) vs spike — no immunological relationship
- Human insulin (P01308) vs spike — no immunological relationship

In [ ]:
# Known cross-reactive and non-cross-reactive pairs
# Each is downloaded from AlphaFold DB and compared against spike

CALIBRATION_PAIRS = {
    "positive_controls": [
        {
            "label": "Strep M protein vs cardiac myosin (rheumatic fever mimicry)",
            "protein_a": {"accession": "P02977", "name": "Streptococcal M protein"},
            "protein_b": {"accession": "P13533", "name": "MYH6 (cardiac myosin)"},
        },
    ],
    "negative_controls": [
        {
            "label": "Hemoglobin beta vs spike (no cross-reactivity)",
            "accession": "P68871",
            "name": "Hemoglobin subunit beta",
        },
        {
            "label": "Insulin vs spike (no cross-reactivity)",
            "accession": "P01308",
            "name": "Insulin",
        },
    ],
}

CAL_DIR = os.path.join(STEP5_DIR, "calibration_structures")
os.makedirs(CAL_DIR, exist_ok=True)

def download_afdb_pdb(accession, dest_dir):
    dest_path = os.path.join(dest_dir, f"{accession}.pdb")
    if os.path.exists(dest_path):
        return dest_path
    resp = requests.get(f"{AFDB_API}/{accession}", timeout=15)
    if resp.status_code == 404:
        print(f"  WARNING: {accession} not found in AlphaFold DB")
        return None
    resp.raise_for_status()
    entries = resp.json()
    if not entries:
        return None
    pdb_resp = requests.get(entries[0]["pdbUrl"], timeout=15)
    pdb_resp.raise_for_status()
    with open(dest_path, "wb") as f:
        f.write(pdb_resp.content)
    return dest_path

# --- Positive controls: compare the two proteins in each pair against each other ---
print("Positive controls (known cross-reactive pairs):\n")
positive_cal_results = []

for pair in CALIBRATION_PAIRS["positive_controls"]:
    path_a = download_afdb_pdb(pair["protein_a"]["accession"], CAL_DIR)
    path_b = download_afdb_pdb(pair["protein_b"]["accession"], CAL_DIR)

    if path_a is None or path_b is None:
        print(f"  {pair['label']}: SKIPPED (download failed)")
        continue

    out_file = os.path.join(CAL_DIR, f"cal_{pair['protein_a']['accession']}_vs_{pair['protein_b']['accession']}.m8")
    cmd = [
        FOLDSEEK_BIN, "easy-search",
        path_a, path_b, out_file, TMP_DIR,
        "--format-output", OUTPUT_COLS,
        "--exhaustive-search", "1",
        "-e", "10",
    ]
    subprocess.run(cmd, capture_output=True, text=True)

    tm = None
    if os.path.exists(out_file) and os.path.getsize(out_file) > 0:
        with open(out_file) as f:
            fields = f.readline().strip().split("\t")
            if len(fields) >= 3:
                tm = float(fields[2])

    positive_cal_results.append({"label": pair["label"], "tm_score": tm})
    print(f"  {pair['label']}: TM={tm:.4f}" if tm else f"  {pair['label']}: no hit")

# --- Negative controls: compare each against spike ---
print("\nNegative controls (no known cross-reactivity, compared vs spike):\n")
negative_cal_results = []

for ctrl in CALIBRATION_PAIRS["negative_controls"]:
    path = download_afdb_pdb(ctrl["accession"], CAL_DIR)
    if path is None:
        print(f"  {ctrl['label']}: SKIPPED (download failed)")
        continue

    for state, spike_pdb in spike_chain_files.items():
        out_file = os.path.join(CAL_DIR, f"cal_{ctrl['accession']}_vs_spike_{state}.m8")
        cmd = [
            FOLDSEEK_BIN, "easy-search",
            spike_pdb, path, out_file, TMP_DIR,
            "--format-output", OUTPUT_COLS,
            "--exhaustive-search", "1",
            "-e", "10",
        ]
        subprocess.run(cmd, capture_output=True, text=True)

        tm = None
        if os.path.exists(out_file) and os.path.getsize(out_file) > 0:
            with open(out_file) as f:
                fields = f.readline().strip().split("\t")
                if len(fields) >= 3:
                    tm = float(fields[2])

        negative_cal_results.append({
            "label": ctrl["label"],
            "spike_state": state,
            "tm_score": tm if tm is not None else 0.0,
        })
        print(f"  {ctrl['name']} vs spike {state}: TM={tm:.4f}" if tm else f"  {ctrl['name']} vs spike {state}: no hit")

# Summary
print("\n--- Calibration summary ---")
pos_tms = [r["tm_score"] for r in positive_cal_results if r["tm_score"] is not None]
neg_tms = [r["tm_score"] for r in negative_cal_results]
if pos_tms:
    print(f"Positive control TM range: [{min(pos_tms):.4f} - {max(pos_tms):.4f}]")
if neg_tms:
    print(f"Negative control TM range: [{min(neg_tms):.4f} - {max(neg_tms):.4f}]")

tpm1_pf_tm = tpm1_postfusion["tm_score"]
print(f"\nTPM1 postfusion observed TM: {tpm1_pf_tm:.4f}")
if pos_tms:
    if tpm1_pf_tm >= min(pos_tms):
        print("  -> Falls WITHIN or ABOVE known cross-reactive pair range")
    else:
        print("  -> Falls BELOW known cross-reactive pair range")

## Combined assessment

A result is retained as a candidate for Step 6 (antigenicity scoring) only if it clears
**both** validation tracks: statistically significant against the size-matched null distribution
AND falling within or above the known cross-reactive pair range.

In [ ]:
# Determine which Step 4 results survive both validation tracks

ALPHA = 0.05  # significance threshold for empirical p-value

retained_hits = []
rejected_hits = []

for p in pvalue_results:
    sig = p["empirical_pvalue"] is not None and p["empirical_pvalue"] < ALPHA
    # Known-pair check: does the observed TM fall within or above the positive control range?
    in_cal_range = pos_tms and p["observed_tm"] >= min(pos_tms) if pos_tms else None

    status = {
        **p,
        "passes_null_test": sig,
        "passes_calibration": in_cal_range,
        "retained": sig and (in_cal_range if in_cal_range is not None else True),
    }

    if status["retained"]:
        retained_hits.append(status)
        print(f"  RETAINED: {p['antigen']} {p['spike_state']} "
              f"(TM={p['observed_tm']:.4f}, p={p['empirical_pvalue']:.4f})")
    else:
        rejected_hits.append(status)

if not retained_hits:
    print("  No results survived both validation tracks.")
    print("  This is a clean, validated null result: spike shares no statistically")
    print("  significant structural similarity with these cardiac autoantigens beyond")
    print("  what is expected by chance, across all three conformational states.")
else:
    print(f"\n{len(retained_hits)} result(s) retained for Step 6 antigenicity scoring.")
    print(f"{len(rejected_hits)} result(s) rejected (did not clear both validation tracks).")

In [ ]:
from datetime import datetime, timezone

validation_output = {
    "pipeline_step": 5,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "generated_in": "Google Colab",
    "null_distribution": {
        "n_random_proteins": len(downloaded_null),
        "n_failed_downloads": len(failed_downloads),
        "n_total_comparisons": len(null_scores),
        "size_bins": SIZE_BINS,
        "antigen_size_bin_mapping": ANTIGEN_SIZE_BIN,
    },
    "calibration": {
        "positive_controls": positive_cal_results,
        "negative_controls": negative_cal_results,
    },
    "pvalue_results": pvalue_results,
    "significance_threshold": ALPHA,
    "retained_hits": retained_hits,
    "rejected_hits": rejected_hits,
    "interpretation": (
        "Results listed in 'retained_hits' cleared both the empirical p-value test "
        "(against a size-matched null distribution of random human proteins) and the "
        "known cross-reactive pair calibration check. These proceed to Step 6 for "
        "antigenicity and solvent accessibility scoring. Results in 'rejected_hits' "
        "did not clear one or both tracks and are not carried forward, though they "
        "are recorded here for full transparency."
    ),
}

output_path = os.path.join(STEP5_DIR, "validation_results.json")
with open(output_path, "w") as f:
    json.dump(validation_output, f, indent=2, default=str)

print("validation_results.json written to:", output_path)

---

### Before moving to Step 6

If `retained_hits` is empty, the study has produced a validated null result across all antigens
and all conformational states. This is a complete, publishable finding: spike and these cardiac
autoantigens share no statistically significant structural similarity, and the conformational
ensemble approach does not rescue the mimicry hypothesis. The methods section and Discussion
are already written to frame this outcome as a contribution, not a failure.

If one or more results survived (most likely TPM1 postfusion), those proceed to Step 6 for
antigenicity and solvent accessibility scoring via ElliPro, which tests whether the structurally
similar region is actually antibody-accessible or buried and non-immunogenic.